# 05 Cohort Retention Analysis

This notebook analyzes customer retention using cohort analysis.

A cohort is defined by the month of a customer's first product purchase. Retention is measured by whether customer from the same cohort return and purchase again in later months.

Scope:
- Build customer-month purchase table
- Identify each customer's first purchase month
- Calculate cohort index
- Build cohort retention table
- Export retention outputs

In [9]:
import pandas as pd
import numpy as np

In [10]:
clean_product_sales = pd.read_csv("../data/processed/clean_product_sales.csv")

clean_product_sales["invoice_date"] = pd.to_datetime(clean_product_sales["invoice_date"])

clean_product_sales.head()

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,is_cancelled,revenue,is_negative_quantity,is_zero_quantity,is_positive_sale,is_non_product_transaction,invoice_year,invoice_month,invoice_day,invoice_hour,day_of_week
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0,False,False,True,False,2009,2009-12,2009-12-01,7,Tuesday


In [11]:
cohort_base = clean_product_sales.dropna(subset=['customer_id']).copy()

cohort_base['customer_id'] = (
    cohort_base['customer_id'].astype('int64').astype('str')
)

cohort_base['order_month'] = cohort_base['invoice_date'].dt.to_period('M').dt.to_timestamp()

cohort_base[['customer_id', 'invoice', 'invoice_date', 'order_month', 'revenue']].head()

,customer_id,invoice,invoice_date,order_month,revenue
0,13085,489434,2009-12-01 07:45:00,2009-12-01,83.4
1,13085,489434,2009-12-01 07:45:00,2009-12-01,81.0
2,13085,489434,2009-12-01 07:45:00,2009-12-01,81.0
3,13085,489434,2009-12-01 07:45:00,2009-12-01,100.8
4,13085,489434,2009-12-01 07:45:00,2009-12-01,30.0


In [12]:
#indentify cohort month
customer_first_purchase = (
    cohort_base
    .groupby('customer_id')
    .agg(cohort_month=('order_month', 'min'))
    .reset_index()
)

cohort_data = cohort_base.merge(
    customer_first_purchase, on='customer_id', how='left'
)

cohort_data[['customer_id', 'invoice', 'order_month', 'cohort_month']].head()

,customer_id,invoice,order_month,cohort_month
0,13085,489434,2009-12-01,2009-12-01
1,13085,489434,2009-12-01,2009-12-01
2,13085,489434,2009-12-01,2009-12-01
3,13085,489434,2009-12-01,2009-12-01
4,13085,489434,2009-12-01,2009-12-01


In [13]:
#calculate cohort index
cohort_data['cohort_index'] = (
    (cohort_data['order_month'].dt.year - cohort_data['cohort_month'].dt.year) * 12 +
    (cohort_data['order_month'].dt.month - cohort_data['cohort_month'].dt.month) + 1
)

cohort_data[['customer_id', 'order_month', 'cohort_month', 'cohort_index']].head()

,customer_id,order_month,cohort_month,cohort_index
0,13085,2009-12-01,2009-12-01,1
1,13085,2009-12-01,2009-12-01,1
2,13085,2009-12-01,2009-12-01,1
3,13085,2009-12-01,2009-12-01,1
4,13085,2009-12-01,2009-12-01,1


In [14]:
#build cohort customer count table
cohort_counts = (
    cohort_data
    .groupby(['cohort_month', 'cohort_index'])
    .agg(customers=('customer_id', 'nunique'))
    .reset_index()
)

cohort_counts.head()

,cohort_month,cohort_index,customers
0,2009-12-01,1,952
1,2009-12-01,2,336
2,2009-12-01,3,317
3,2009-12-01,4,405
4,2009-12-01,5,360


In [15]:
cohort_customer_matrix = cohort_counts.pivot(
    index='cohort_month',
    columns='cohort_index',
    values='customers'
)

cohort_customer_matrix.head()

cohort_index,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,25
cohort_month,,,,,,,,,,,,,,,,,,,,,
2009-12-01,952.0,336.0,317.0,405.0,360.0,342.0,359.0,327.0,321.0,344.0,...,288.0,250.0,288.0,270.0,246.0,242.0,298.0,289.0,386.0,187.0
2010-01-01,382.0,79.0,119.0,117.0,101.0,115.0,99.0,88.0,106.0,121.0,...,57.0,90.0,75.0,71.0,75.0,93.0,74.0,94.0,21.0,NaN
2010-02-01,375.0,88.0,85.0,110.0,92.0,74.0,72.0,108.0,96.0,104.0,...,75.0,60.0,61.0,54.0,86.0,86.0,62.0,22.0,NaN,NaN
2010-03-01,439.0,84.0,102.0,106.0,102.0,90.0,108.0,134.0,122.0,48.0,...,75.0,77.0,69.0,78.0,89.0,94.0,34.0,NaN,NaN,NaN
2010-04-01,293.0,56.0,56.0,47.0,54.0,65.0,81.0,77.0,31.0,32.0,...,46.0,41.0,44.0,53.0,66.0,17.0,NaN,NaN,NaN,NaN


In [16]:
#build retention rate table
cohort_size = cohort_customer_matrix[1]
cohort_retention = cohort_customer_matrix.divide(cohort_size, axis=0)
cohort_retention.head()

cohort_index,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,25
cohort_month,,,,,,,,,,,,,,,,,,,,,
2009-12-01,1.0,0.352941,0.332983,0.425420,0.378151,0.359244,0.377101,0.343487,0.337185,0.361345,...,0.302521,0.262605,0.302521,0.283613,0.258403,0.254202,0.313025,0.303571,0.405462,0.196429
2010-01-01,1.0,0.206806,0.311518,0.306283,0.264398,0.301047,0.259162,0.230366,0.277487,0.316754,...,0.149215,0.235602,0.196335,0.185864,0.196335,0.243455,0.193717,0.246073,0.054974,NaN
2010-02-01,1.0,0.234667,0.226667,0.293333,0.245333,0.197333,0.192000,0.288000,0.256000,0.277333,...,0.200000,0.160000,0.162667,0.144000,0.229333,0.229333,0.165333,0.058667,NaN,NaN
2010-03-01,1.0,0.191344,0.232346,0.241458,0.232346,0.205011,0.246014,0.305239,0.277904,0.109339,...,0.170843,0.175399,0.157175,0.177677,0.202733,0.214123,0.077449,NaN,NaN,NaN
2010-04-01,1.0,0.191126,0.191126,0.160410,0.184300,0.221843,0.276451,0.262799,0.105802,0.109215,...,0.156997,0.139932,0.150171,0.180887,0.225256,0.058020,NaN,NaN,NaN,NaN


## Cohort Retention Interpretation

Each row represents a customer cohort based on the customer's first observed product purchase month. Each column represents the number of months since the first purchase.

Retention rate is calculated as the number of active customers in a later month divided by the original cohort size.

Retention rates may fluctuate across months because this analysis measures monthly repeat purchase activity, not cumulative retention. A customer may be inactive in one month and return in a later month.

Later cohorts have fewer observable months because the dataset ends in 2011-12. Missing retention values for recent cohorts indicate insufficient observation time, not necessarily poor retention. Therefore, mature cohorts and recent cohorts should not be directly compared across long-term retention windows.

In [17]:
#retention table
cohort_retention_formatted = cohort_retention.copy()
cohort_retention_formatted.index = cohort_retention_formatted.index.strftime('%Y-%m')
cohort_retention_formatted = cohort_retention_formatted.map(
    lambda x: f"{x:.2%}" if pd.notna(x) else ""
)
cohort_retention_formatted.head()

cohort_index,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,25
cohort_month,,,,,,,,,,,,,,,,,,,,,
2009-12,100.00%,35.29%,33.30%,42.54%,37.82%,35.92%,37.71%,34.35%,33.72%,36.13%,...,30.25%,26.26%,30.25%,28.36%,25.84%,25.42%,31.30%,30.36%,40.55%,19.64%
2010-01,100.00%,20.68%,31.15%,30.63%,26.44%,30.10%,25.92%,23.04%,27.75%,31.68%,...,14.92%,23.56%,19.63%,18.59%,19.63%,24.35%,19.37%,24.61%,5.50%,
2010-02,100.00%,23.47%,22.67%,29.33%,24.53%,19.73%,19.20%,28.80%,25.60%,27.73%,...,20.00%,16.00%,16.27%,14.40%,22.93%,22.93%,16.53%,5.87%,,
2010-03,100.00%,19.13%,23.23%,24.15%,23.23%,20.50%,24.60%,30.52%,27.79%,10.93%,...,17.08%,17.54%,15.72%,17.77%,20.27%,21.41%,7.74%,,,
2010-04,100.00%,19.11%,19.11%,16.04%,18.43%,22.18%,27.65%,26.28%,10.58%,10.92%,...,15.70%,13.99%,15.02%,18.09%,22.53%,5.80%,,,,


In [18]:
cohort_summary = pd.DataFrame({
    'cohort_month': cohort_customer_matrix.index,
    'cohort_size': cohort_customer_matrix.iloc[:, 0],
    'month_2_retention': cohort_retention.get(2),
    "month_3_retention": cohort_retention.get(3),
    "month_6_retention": cohort_retention.get(6),
    "month_12_retention": cohort_retention.get(12)
}).reset_index(drop=True)

cohort_summary['cohort_month'] = cohort_summary['cohort_month'].dt.strftime('%Y-%m')
cohort_summary

,cohort_month,cohort_size,month_2_retention,month_3_retention,month_6_retention,month_12_retention
0,2009-12,952.0,0.352941,0.332983,0.359244,0.495798
1,2010-01,382.0,0.206806,0.311518,0.301047,0.172775
2,2010-02,375.0,0.234667,0.226667,0.197333,0.125333
3,2010-03,439.0,0.191344,0.232346,0.205011,0.143508
4,2010-04,293.0,0.191126,0.191126,0.221843,0.139932
5,2010-05,254.0,0.157480,0.169291,0.255906,0.133858
6,2010-06,267.0,0.176030,0.187266,0.284644,0.138577
7,2010-07,185.0,0.156757,0.183784,0.140541,0.145946
8,2010-08,162.0,0.197531,0.290123,0.117284,0.117284
9,2010-09,239.0,0.230126,0.234310,0.104603,0.100418


In [19]:
cohort_customer_matrix.to_csv("../data/processed/cohort_customer_matrix.csv")
cohort_retention.to_csv("../data/processed/cohort_retention.csv")
cohort_retention_formatted.to_csv("../data/processed/cohort_retention_formatted.csv")
cohort_summary.to_csv("../data/processed/cohort_summary.csv", index=False)

print("Cohort retention outputs exported successfully.")

Cohort retention outputs exported successfully.
